[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AmolSISL/IMDb-Deep-Learning-Research/blob/main/notebooks/05_RQ3_Anomaly_Detection.ipynb)

# Notebook 05 — RQ3: Autoencoder-Based Review Quality Gate

> **RQ3.** Can autoencoder-based anomaly detection identify low-quality, off-topic, or templated reviews so that they can be screened out before sentiment analysis?

**Design (synopsis Table 3):**
* **Training:** about 20,000 normal IMDb reviews (training and validation splits). The autoencoders only ever see normal reviews.
* **Evaluation set:** 9,500 normal reviews from the official IMDb test set plus **500 injected anomalies** (5% contamination). This exceeds the 385 anomalies needed to estimate recall within ±5%.
* **Calibration set:** a separate 1,900 normal reviews (development test) plus 100 independently generated anomalies. It is used only to choose the threshold, so the evaluation set stays untouched.
* **Anomaly types** (the public data contain no spam labels, so anomalies are injected as a controlled proxy, following Jindal & Liu, 2008):
  1. **Off-domain:** Amazon Electronics reviews posted in a movie-review stream;
  2. **Very short or meaningless text:** gibberish, keyboard mashing, single generic words, repeated words;
  3. **Duplicated templated text:** promotional or incentivised spam built from a few templates.
* **Detectors:** dense autoencoder and variational autoencoder (Kingma & Welling, 2014), each on two representations: **TF-IDF** (5,000 terms) and **sentence embeddings** (all-MiniLM-L6-v2). The anomaly score is the reconstruction error (MSE).
* **Metrics:** precision, recall, F1, AUROC, AUPRC, recall by anomaly type, and the change in downstream sentiment accuracy after filtering.
* **Secondary analysis:** VAE-generated samples used to augment a minority class.

## 0. Setup

In [ ]:
# ---------------------------------------------------------------
# Project setup: mount Google Drive so every notebook shares the same
# data, models and outputs (folder: MyDrive/customer-voice-intelligence)
# ---------------------------------------------------------------
import os, sys, warnings
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/customer-voice-intelligence"
else:  # running locally
    PROJECT_DIR = os.environ.get("CVI_PROJECT_DIR", os.path.abspath("customer-voice-intelligence"))

for sub in ["data/raw", "data/processed", "outputs/figures", "outputs/tables",
            "outputs/results", "models", "code"]:
    os.makedirs(os.path.join(PROJECT_DIR, sub), exist_ok=True)
sys.path.insert(0, os.path.join(PROJECT_DIR, "code"))
print("Project folder:", PROJECT_DIR)

In [ ]:
!pip -q install sentence-transformers
import importlib, json, time, random, string, copy
import cvi_utils as U
importlib.reload(U); U.set_project(PROJECT_DIR)
import numpy as np, pandas as pd, joblib, torch, torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, f1_score
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LogisticRegression
from statsmodels.stats.proportion import proportion_confint
U.plot_style(); U.set_seed(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu"); print("Device:", DEVICE)

## 1. Configuration

In [ ]:
N_NORMAL_TRAIN = 20000
N_EVAL_NORMAL, N_EVAL_ANOM = 9500, 500
N_CAL_NORMAL, N_CAL_ANOM = 1900, 100
TFIDF_FEATURES = 5000
EMB_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
AE_EPOCHS, AE_PATIENCE, AE_LR, AE_BATCH = 40, 4, 1e-3, 256
SEEDS_AUG = [42, 43, 44, 45, 46]
if U.QUICK:
    N_NORMAL_TRAIN, N_EVAL_NORMAL, N_EVAL_ANOM, N_CAL_NORMAL, N_CAL_ANOM, AE_EPOCHS = 500, 380, 20, 150, 9, 3
    SEEDS_AUG = [42, 43]
RES = U.P("outputs", "results"); MOD = U.P("models", "rq3"); os.makedirs(MOD, exist_ok=True)

## 2. Normal data and injected anomalies

In [ ]:
imdb, amz = U.load_imdb(), U.load_amazon()
normal_train = imdb[imdb.split.isin(["train", "validation"])].sample(n=min(N_NORMAL_TRAIN, int(imdb.split.isin(["train", "validation"]).sum())), random_state=42)
eval_normal = imdb[imdb.split == "test"].sample(n=N_EVAL_NORMAL, random_state=42)
cal_normal = imdb[imdb.split == "dev_test"].sample(n=N_CAL_NORMAL, random_state=42)

ADJ = ["amazing", "awesome", "incredible", "fantastic", "perfect", "unbelievable", "outstanding", "brilliant"]
TITLES = ["the new blockbuster", "this film", "Movie 2", "the latest release", "this masterpiece", "the sequel"]
SITES = ["www.free-hd-movies.example", "watchnow.example", "bestflix.example", "streamfree.example"]
TEMPLATES = [
    "Watch {title} full movie free online at {site}. No signup, HD quality, click the link now!",
    "Best movie ever!!! {adj} {adj2} film. Highly recommend to everyone. Five stars, must watch, trust me.",
    "I received a free screener of this film in exchange for my honest review. {Adj} film, {adj} acting, {adj2} story. Recommended.",
    "This is the most {adj} movie of {year}. Everyone must see it. {Adj}! {Adj}! Go watch it today.",
    "Great movie. Great acting. Great story. Great music. Great everything. {Adj}. Recommended to all my friends.",
    "Download {title} now at {site} for free, the {adj} movie everyone is talking about!!!",
    "Worst movie ever. Do not watch. Total waste. Avoid avoid avoid. Visit {site} for better films.",
    "Amazing film, amazing cast, amazing director, amazing {adj2} ending. 10 out of 10 for sure!!!",
]
WORDS = ["good", "nice", "ok", "movie", "great", "bad", "fine", "wow", "lol", "yes", "meh"]

def make_short(rng):
    kind = rng.randrange(5)
    if kind == 0:    # random letters
        return " ".join("".join(rng.choices(string.ascii_lowercase, k=rng.randint(3, 9))) for _ in range(rng.randint(1, 6)))
    if kind == 1:    # keyboard mashing
        return rng.choice(["asdf", "qwerty", "jkl;", "zxcv", "hjkl"]) * rng.randint(1, 4)
    if kind == 2:    # single generic word
        return rng.choice(WORDS) + rng.choice(["", ".", "!", "!!"])
    if kind == 3:    # repeated word
        return " ".join([rng.choice(WORDS)] * rng.randint(3, 12))
    return "".join(rng.choices("!?.*#@ ", k=rng.randint(3, 15)))   # punctuation noise

def make_template(rng):
    a, a2 = rng.sample(ADJ, 2)
    return rng.choice(TEMPLATES).format(title=rng.choice(TITLES), site=rng.choice(SITES), adj=a, adj2=a2,
                                        Adj=a.capitalize(), year=rng.randint(2005, 2026))

def make_anomalies(n, amazon_df, seed, prefix):
    rng = random.Random(seed)
    k = [n // 3 + (1 if i < n % 3 else 0) for i in range(3)]
    off = amazon_df.sample(n=k[0], random_state=seed)
    rows = [dict(review_id=f"{prefix}_off_{i}", review_text=t, label=int(l), anomaly_type="off_domain") for i, (t, l) in enumerate(zip(off.review_text, off.label))]
    rows += [dict(review_id=f"{prefix}_short_{i}", review_text=make_short(rng), label=rng.randint(0, 1), anomaly_type="short_meaningless") for i in range(k[1])]
    rows += [dict(review_id=f"{prefix}_tmpl_{i}", review_text=make_template(rng), label=rng.randint(0, 1), anomaly_type="templated") for i in range(k[2])]
    return pd.DataFrame(rows).assign(is_anomaly=1)

def assemble(normal, anomalies):
    n = normal[["review_id", "review_text", "label"]].assign(anomaly_type="normal", is_anomaly=0)
    return pd.concat([n, anomalies], ignore_index=True).sample(frac=1, random_state=0).reset_index(drop=True)

eval_set = assemble(eval_normal, make_anomalies(N_EVAL_ANOM, amz[amz.split == "test"], seed=1, prefix="eval"))
cal_set = assemble(cal_normal, make_anomalies(N_CAL_ANOM, amz[amz.split == "finetune_pool"], seed=2, prefix="cal"))
eval_set.to_csv(U.P("data", "processed", "rq3_eval_set.csv"), index=False)
print("Evaluation set:", eval_set.anomaly_type.value_counts().to_dict())
print("Calibration set:", cal_set.anomaly_type.value_counts().to_dict())
eval_set[eval_set.is_anomaly == 1].groupby("anomaly_type").head(2)[["anomaly_type", "review_text"]]

## 3. Representations
Both representations are fitted on the normal training reviews only.

In [ ]:
tfidf = TfidfVectorizer(tokenizer=U.tokenize, token_pattern=None, lowercase=False, max_features=TFIDF_FEATURES,
                        min_df=5 if not U.QUICK else 2, sublinear_tf=True, dtype=np.float32)
R = {"tfidf": {}, "embedding": {}}
R["tfidf"]["train"] = tfidf.fit_transform(normal_train.review_text).toarray()
R["tfidf"]["eval"] = tfidf.transform(eval_set.review_text).toarray()
R["tfidf"]["cal"] = tfidf.transform(cal_set.review_text).toarray()
joblib.dump(tfidf, f"{MOD}/tfidf_5000.joblib")

from sentence_transformers import SentenceTransformer
st = SentenceTransformer(EMB_MODEL, device=str(DEVICE))
enc = lambda s: st.encode(list(s), batch_size=128, show_progress_bar=False, convert_to_numpy=True, normalize_embeddings=True)
emb = {"train": enc(normal_train.review_text), "eval": enc(eval_set.review_text), "cal": enc(cal_set.review_text)}
scaler = StandardScaler().fit(emb["train"])
R["embedding"] = {k: scaler.transform(v).astype(np.float32) for k, v in emb.items()}
for r in R:
    print(r, {k: v.shape for k, v in R[r].items()})
print("All-zero TF-IDF rows (no known word) in evaluation set:", int((R["tfidf"]["eval"].sum(1) == 0).sum()))

## 4. Autoencoder and variational autoencoder

In [ ]:
class AE(nn.Module):
    def __init__(self, d, h=256, z=32):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(d, h), nn.ReLU(), nn.Linear(h, z), nn.ReLU())
        self.dec = nn.Sequential(nn.Linear(z, h), nn.ReLU(), nn.Linear(h, d))
    def forward(self, x):
        return self.dec(self.enc(x)), None

class VAE(nn.Module):
    def __init__(self, d, h=256, z=16):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(d, h), nn.ReLU())
        self.mu, self.logvar = nn.Linear(h, z), nn.Linear(h, z)
        self.dec = nn.Sequential(nn.Linear(z, h), nn.ReLU(), nn.Linear(h, d))
        self.z = z
    def forward(self, x):
        h = self.enc(x); mu, logvar = self.mu(h), self.logvar(h).clamp(-10, 10)
        zz = mu + torch.randn_like(mu) * torch.exp(0.5 * logvar) if self.training else mu
        return self.dec(zz), (mu, logvar)

def loss_fn(x, xh, extra, beta):
    rec = ((xh - x) ** 2).sum(1)                                     # summed squared error per review
    if extra is None:
        return rec.mean()
    mu, logvar = extra
    kl = -0.5 * (1 + logvar - mu ** 2 - logvar.exp()).sum(1)
    return (rec + beta * kl).mean()

def train_ae(kind, X, seed=42, beta=1.0):
    U.set_seed(seed)
    rs = np.random.RandomState(seed); perm = rs.permutation(len(X)); nv = len(X) // 10
    Xv, Xt = torch.tensor(X[perm[:nv]]), torch.tensor(X[perm[nv:]])
    model = (AE if kind == "ae" else VAE)(X.shape[1]).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=AE_LR)
    # scale the reconstruction term so that KL and reconstruction are comparable for small-valued TF-IDF inputs
    best, best_state, bad, hist = np.inf, None, 0, []
    for ep in range(AE_EPOCHS):
        model.train(); idx = torch.randperm(len(Xt))
        for i in range(0, len(Xt), AE_BATCH):
            xb = Xt[idx[i:i + AE_BATCH]].to(DEVICE)
            xh, extra = model(xb); loss = loss_fn(xb, xh, extra, beta)
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            xh, extra = model(Xv.to(DEVICE)); vl = loss_fn(Xv.to(DEVICE), xh, extra, beta).item()
        hist.append(vl)
        if vl < best - 1e-6:
            best, bad, best_state = vl, 0, copy.deepcopy(model.state_dict())
        else:
            bad += 1
            if bad >= AE_PATIENCE: break
    model.load_state_dict(best_state)
    return model, hist

@torch.no_grad()
def recon_error(model, X, batch=2048):
    model.eval(); out = []
    for i in range(0, len(X), batch):
        xb = torch.tensor(X[i:i + batch]).to(DEVICE); xh, _ = model(xb)
        out.append(((xh - xb) ** 2).mean(1).cpu().numpy())
    return np.concatenate(out)

DETECTORS = {}
for rep in ["tfidf", "embedding"]:
    for kind in ["ae", "vae"]:
        name = f"{kind.upper()} / {'TF-IDF' if rep == 'tfidf' else 'Embedding'}"
        beta = 1.0 if rep == "embedding" else 0.01            # TF-IDF reconstruction errors are tiny; weaker KL keeps the decoder informative
        with U.Timer() as t:
            model, hist = train_ae(kind, R[rep]["train"], beta=beta)
        DETECTORS[name] = dict(rep=rep, kind=kind, model=model, epochs=len(hist), minutes=t.seconds / 60,
                               train_err=recon_error(model, R[rep]["train"]),
                               cal=recon_error(model, R[rep]["cal"]), eval=recon_error(model, R[rep]["eval"]))
        torch.save(model.state_dict(), f"{MOD}/{kind}_{rep}.pt")
        print(f"{name:20s} epochs {len(hist):2d}  {t.seconds:.0f}s  "
              f"calibration AUROC {roc_auc_score(cal_set.is_anomaly, DETECTORS[name]['cal']):.3f}")

## 5. Threshold selection and evaluation
Two thresholds are compared for each detector:
* **Calibrated:** the reconstruction error that maximises F1 on the calibration set. Choosing it requires a small labelled sample, which a quality team can produce by inspecting ~2,000 reviews.
* **Unsupervised:** the 95th percentile of the training error. This needs no labels and assumes about 5% contamination.

All metrics are computed on the evaluation set, which the threshold choice never sees.

In [ ]:
def best_f1_threshold(y, s):
    p, r, t = precision_recall_curve(y, s)
    f = 2 * p * r / np.clip(p + r, 1e-12, None)
    return t[np.argmax(f[:-1])]

def wilson(k, n):
    lo, hi = proportion_confint(k, n, method="wilson"); return lo, hi

y_eval, y_cal = eval_set.is_anomaly.to_numpy(), cal_set.is_anomaly.to_numpy()
rows = []
for name, d in DETECTORS.items():
    for thr_name, thr in [("calibrated (max F1)", best_f1_threshold(y_cal, d["cal"])), ("95th pct of training error", np.percentile(d["train_err"], 95))]:
        flag = (d["eval"] >= thr).astype(int)
        tp = int((flag & y_eval).sum()); rec = tp / y_eval.sum(); lo, hi = wilson(tp, y_eval.sum())
        rows.append(dict(detector=name, threshold_rule=thr_name, threshold=thr,
                         precision=tp / max(flag.sum(), 1), recall=rec, recall_ci=f"[{lo:.2f}, {hi:.2f}]",
                         f1=f1_score(y_eval, flag), auroc=roc_auc_score(y_eval, d["eval"]), auprc=average_precision_score(y_eval, d["eval"]),
                         flagged_share=flag.mean(), normal_removed=(flag[y_eval == 0]).mean(), epochs=d["epochs"], train_min=d["minutes"]))
det = pd.DataFrame(rows)
det.to_csv(f"{RES}/rq3_detector_results.csv", index=False)

cal_auprc = {n: average_precision_score(y_cal, d["cal"]) for n, d in DETECTORS.items()}
BEST = max(cal_auprc, key=cal_auprc.get)
print("Best detector by calibration AUPRC:", BEST)

table12 = det.assign(**{c: det[c].map("{:.3f}".format) for c in ["precision", "recall", "f1", "auroc", "auprc"]},
                     flagged_share=(100 * det.flagged_share).map("{:.1f}%".format), normal_removed=(100 * det.normal_removed).map("{:.1f}%".format))
table12 = table12[["detector", "threshold_rule", "precision", "recall", "recall_ci", "f1", "auroc", "auprc", "flagged_share", "normal_removed"]]
table12.columns = ["Detector", "Threshold", "Precision", "Recall", "Recall 95% CI", "F1", "AUROC", "AUPRC", "Flagged", "Normal reviews removed"]
U.save_table(table12, "table12_rq3_detection")
table12

### Table 13 — Recall by anomaly type (calibrated thresholds)

In [ ]:
types = ["off_domain", "short_meaningless", "templated"]
rows = []
for name, d in DETECTORS.items():
    thr = best_f1_threshold(y_cal, d["cal"]); flag = d["eval"] >= thr
    r = {"Detector": name}
    for t in types:
        m = (eval_set.anomaly_type == t).to_numpy(); r[t.replace("_", " ").capitalize()] = f"{flag[m].mean():.2f}"
    r["Normal (false-positive rate)"] = f"{flag[(y_eval == 0)].mean():.3f}"
    rows.append(r)
table13 = pd.DataFrame(rows); U.save_table(table13, "table13_rq3_recall_by_type"); table13

### Complementary rule: near-duplicate detection
Templated spam is fluent and on-topic, so it can reconstruct well. Its defining property is that it **repeats**: Jindal and Liu (2008) report that near-duplicates are the most common form of opinion spam. A near-duplicate rule flags a review when another review in the same batch has TF-IDF cosine similarity ≥ 0.8. The rule is combined with the best autoencoder in a **hybrid gate** (flag if either fires).

In [ ]:
DUP_SIM = 0.8
X_eval_tfidf = tfidf.transform(eval_set.review_text)
nn_ = NearestNeighbors(n_neighbors=2, metric="cosine").fit(X_eval_tfidf)
dist, _ = nn_.kneighbors(X_eval_tfidf)
nonzero = np.asarray(X_eval_tfidf.sum(1)).ravel() > 0
dup_flag = ((1 - dist[:, 1]) >= DUP_SIM) & nonzero

best_d = DETECTORS[BEST]; thr_best = best_f1_threshold(y_cal, best_d["cal"])
ae_flag = best_d["eval"] >= thr_best
GATES = {f"{BEST} only": ae_flag, "Near-duplicate rule only": dup_flag, f"Hybrid gate ({BEST} OR near-duplicate)": ae_flag | dup_flag}
rows = []
for g, f in GATES.items():
    f = f.astype(int); tp = int((f & y_eval).sum())
    r = {"Gate": g, "Precision": f"{tp / max(f.sum(), 1):.3f}", "Recall": f"{tp / y_eval.sum():.3f}", "F1": f"{f1_score(y_eval, f):.3f}",
         "Normal reviews removed": f"{100 * f[y_eval == 0].mean():.1f}%"}
    for t in types:
        r[f"Recall: {t}"] = f"{f[(eval_set.anomaly_type == t).to_numpy()].mean():.2f}"
    rows.append(r)
table13b = pd.DataFrame(rows); U.save_table(table13b, "table13b_rq3_hybrid_gate"); table13b

### Figures 13 and 14 — Score distributions and precision-recall curves

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4))
col = {"normal": "0.6", "off_domain": "#e09f3e", "short_meaningless": U.NEG_COLOR, "templated": "#9c4f96"}
s = best_d["eval"]; bins = np.logspace(np.log10(max(s.min(), 1e-8)), np.log10(s.max()), 60)
for t in ["normal"] + types:
    m = (eval_set.anomaly_type == t).to_numpy()
    ax.hist(s[m], bins=bins, alpha=0.6, density=True, color=col[t], label=f"{t.replace('_', ' ')} (n = {m.sum():,})")
ax.axvline(thr_best, color="k", ls="--", lw=1, label="Calibrated threshold")
ax.set_xscale("log"); ax.set_xlabel("Reconstruction error (MSE, log scale)"); ax.set_ylabel("Density")
ax.set_title(f"Figure 13. Reconstruction error by review type — {BEST}"); ax.legend(fontsize=8)
U.save_fig(fig, "fig13_recon_error_distribution"); plt.show()

fig, ax = plt.subplots(figsize=(6.5, 4.5))
for name, d in DETECTORS.items():
    p, r, _ = precision_recall_curve(y_eval, d["eval"])
    ax.plot(r, p, label=f"{name} (AUPRC {average_precision_score(y_eval, d['eval']):.2f})")
ax.axhline(y_eval.mean(), color="0.5", ls=":", label=f"Random ({y_eval.mean():.0%} anomalies)")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision"); ax.set_title("Figure 14. Precision-recall curves on the evaluation set"); ax.legend(fontsize=8)
U.save_fig(fig, "fig14_pr_curves"); plt.show()

## 6. Downstream effect on sentiment analysis
The TF-IDF + logistic-regression model from RQ1 scores every review in the evaluation set. Each anomaly carries the label it would contribute to a dashboard: the original star label for off-domain reviews, and a random label for meaningless or templated text, which simulates manipulated ratings. Two outcomes are reported:
* **sentiment accuracy** over the reviews that pass the gate;
* the **dashboard error**: the gap between the predicted share of positive reviews and the true share of positive reviews among genuine (normal) reviews.

In [ ]:
vec = joblib.load(U.P("models", "rq1", "tfidf_raw.joblib")); lr = joblib.load(U.P("models", "rq1", "logreg_raw_s42.joblib"))
p_sent = lr.predict_proba(vec.transform(eval_set.review_text))[:, 1]
pred = (p_sent >= 0.5).astype(int); yl = eval_set.label.to_numpy()
true_share = yl[y_eval == 0].mean()

def outcome(keep, name):
    return {"Scenario": name, "Reviews kept": int(keep.sum()), "Anomalies kept": int((keep & (y_eval == 1)).sum()),
            "Sentiment accuracy": (pred[keep] == yl[keep]).mean(),
            "Predicted positive share": pred[keep].mean(), "Dashboard error (pp)": 100 * abs(pred[keep].mean() - true_share)}

rows = [outcome(np.ones_like(y_eval, bool), "No gate"),
        outcome(~ae_flag, f"Autoencoder gate ({BEST})"),
        outcome(~(ae_flag | dup_flag), "Hybrid gate"),
        outcome(y_eval == 0, "Oracle (all anomalies removed)")]
table14 = pd.DataFrame(rows)
table14["Accuracy change vs no gate (pp)"] = 100 * (table14["Sentiment accuracy"] - table14["Sentiment accuracy"].iloc[0])
for c in ["Sentiment accuracy", "Predicted positive share"]:
    table14[c] = (100 * table14[c]).map("{:.2f}%".format)
table14["Dashboard error (pp)"] = table14["Dashboard error (pp)"].map("{:.2f}".format)
table14["Accuracy change vs no gate (pp)"] = table14["Accuracy change vs no gate (pp)"].map("{:+.2f}".format)
U.save_table(table14, "table14_rq3_downstream")
print(f"True positive share among genuine reviews: {100 * true_share:.2f}%")
print("Sentiment accuracy on each anomaly type (no gate):",
      {t: round(float((pred[(eval_set.anomaly_type == t).to_numpy()] == yl[(eval_set.anomaly_type == t).to_numpy()]).mean()), 3) for t in types})
table14

In [ ]:
rec = {"best_detector": BEST, "calibrated_threshold": float(thr_best),
       "percentile_threshold": float(np.percentile(best_d["train_err"], 95)), "dup_similarity": DUP_SIM}
rec.update(det[(det.detector == BEST) & (det.threshold_rule.str.startswith("calibrated"))].iloc[0][["precision", "recall", "f1", "auroc", "auprc"]].astype(float).to_dict())
# gate latency: raw text -> representation -> reconstruction error (CPU, 2 threads, 1,000 reviews)
lat_texts = eval_set.review_text.tolist()[:1000]
cpu_model = copy.deepcopy(best_d["model"]).cpu().eval(); nthr = torch.get_num_threads(); torch.set_num_threads(2)
def gate_fn():
    Xr = tfidf.transform(lat_texts).toarray() if best_d["rep"] == "tfidf" else scaler.transform(st.encode(lat_texts, batch_size=128, normalize_embeddings=True, device="cpu")).astype(np.float32)
    with torch.no_grad():
        xb = torch.tensor(Xr); xh, _ = cpu_model(xb); return ((xh - xb) ** 2).mean(1)
gate_fn()
with U.Timer() as t: gate_fn()
torch.set_num_threads(nthr)
rec["gate_cpu_ms_per_review"] = 1000 * t.seconds / len(lat_texts)
hy = (ae_flag | dup_flag).astype(int)
rec.update(hybrid_precision=float((hy & y_eval).sum() / max(hy.sum(), 1)), hybrid_recall=float((hy & y_eval).sum() / y_eval.sum()),
           hybrid_normal_removed=float(hy[y_eval == 0].mean()),
           acc_no_gate=float((pred == yl).mean()), acc_hybrid_gate=float((pred[hy == 0] == yl[hy == 0]).mean()),
           dashboard_error_no_gate_pp=float(100 * abs(pred.mean() - true_share)),
           dashboard_error_hybrid_pp=float(100 * abs(pred[hy == 0].mean() - true_share)))
print(f"Gate latency on CPU: {rec['gate_cpu_ms_per_review']:.2f} ms per review")
json.dump(rec, open(f"{RES}/rq3_decision.json", "w"), indent=2)
rec

## 7. Secondary analysis: VAE augmentation of a minority class
**Scenario:** a new category where negative reviews are rare (10% of the training data, 500 negatives and 4,500 positives). A VAE trained **only on the 500 negative reviews** generates synthetic negative TF-IDF vectors. Four training sets for logistic regression are compared on the balanced development test set (macro F1, 5 seeds):
1. imbalanced data, no correction;
2. class weighting;
3. random oversampling of negatives;
4. VAE-generated negatives.

Generation happens in the TF-IDF feature space, so the "generated text" is a weighted bag of words. A few samples are decoded into their top words for illustration.

In [ ]:
X_all = tfidf.transform(imdb[imdb.split == "train"].review_text).toarray(); y_all = imdb.loc[imdb.split == "train", "label"].to_numpy()
X_dev = tfidf.transform(imdb[imdb.split == "dev_test"].review_text); y_dev = imdb.loc[imdb.split == "dev_test", "label"].to_numpy()
N_POS, N_NEG = (4500, 500) if not U.QUICK else (200, 25)
vocab_arr = np.array(tfidf.get_feature_names_out())
aug_rows, samples_shown = [], None
for seed in SEEDS_AUG:
    rs = np.random.RandomState(seed)
    pi = rs.choice(np.where(y_all == 1)[0], N_POS, replace=False); ni = rs.choice(np.where(y_all == 0)[0], N_NEG, replace=False)
    Xp, Xn = X_all[pi], X_all[ni]
    def fit_eval(X, yv, **kw):
        m = LogisticRegression(C=4, solver="liblinear", max_iter=2000, **kw).fit(X, yv)
        return f1_score(y_dev, m.predict(X_dev), average="macro")
    base_X = np.vstack([Xp, Xn]); base_y = np.r_[np.ones(N_POS), np.zeros(N_NEG)]
    r = {"seed": seed, "Imbalanced (no correction)": fit_eval(base_X, base_y),
         "Class weighting": fit_eval(base_X, base_y, class_weight="balanced")}
    over = Xn[rs.choice(N_NEG, N_POS - N_NEG, replace=True)]
    r["Random oversampling"] = fit_eval(np.vstack([base_X, over]), np.r_[base_y, np.zeros(len(over))])
    vae, _ = train_ae("vae", Xn.astype(np.float32), seed=seed, beta=0.01)
    vae.eval()
    with torch.no_grad():
        z = torch.randn(N_POS - N_NEG, vae.z, device=DEVICE); synth = vae.dec(z).cpu().numpy()
    synth = np.clip(synth, 0, None); synth /= np.linalg.norm(synth, axis=1, keepdims=True) + 1e-12   # same scale as TF-IDF rows
    r["VAE-generated negatives"] = fit_eval(np.vstack([base_X, synth]), np.r_[base_y, np.zeros(len(synth))])
    aug_rows.append(r)
    if samples_shown is None:
        samples_shown = [", ".join(vocab_arr[np.argsort(-s)[:12]]) for s in synth[:5]]
aug = pd.DataFrame(aug_rows)
table15 = aug.drop(columns="seed").agg(["mean", "std"]).T
table15["Macro F1 (mean ± SD)"] = [f"{m:.4f} ± {s:.4f}" for m, s in zip(table15["mean"], table15["std"])]
table15["Gain vs imbalanced (pp)"] = (100 * (table15["mean"] - table15.loc["Imbalanced (no correction)", "mean"])).map("{:+.2f}".format)
table15 = table15[["Macro F1 (mean ± SD)", "Gain vs imbalanced (pp)"]]
U.save_table(table15, "table15_rq3_vae_augmentation", index=True)
print("Top words of five VAE-generated negative 'reviews':")
for s in samples_shown: print("  •", s)
table15

## Summary for the report
* **Table 12 / Figure 14:** detection quality of the four autoencoder detectors under two threshold rules.
* **Table 13 / Figure 13:** which anomaly types each detector catches.
* **Table 13b:** the hybrid gate (autoencoder plus near-duplicate rule) as the recommended **review quality gate**.
* **Table 14:** how much the gate protects sentiment accuracy and the dashboard positive share.
* **Table 15:** whether VAE augmentation helps a minority-class scenario.
* **Limitation:** the anomalies are injected, so these results are a controlled proxy for real spam (stated in the report).

**Next:** `06_RQ4_Foundation_Models.ipynb`.